In [ ]:
#Upload kaggle.json
from google.colab import files
files.upload()

In [ ]:
#Set up Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
#Download dataset
!kaggle datasets download -d ubiratanfilho/sds-dataset

In [ ]:
#Unzip dataset
!unzip -q sds-dataset.zip

In [ ]:
#Inspect folder contents
!ls -R /content | head -100

In [ ]:
#Function for coco to yolo format conversion
def convert_coco_to_yolo(json_path, images_dir, labels_dir):
    import os, json
    from tqdm import tqdm

    os.makedirs(labels_dir, exist_ok=True)

    with open(json_path) as f:
        data = json.load(f)

    images = {img["id"]: img for img in data["images"]}

    valid_categories = [cat for cat in data["categories"] if cat["id"] != 0]

    categories = {cat["id"]: i for i, cat in enumerate(valid_categories)}

    annotations_per_image = {}
    for ann in data["annotations"]:
        if ann["category_id"] == 0:
            continue

        img_id = ann["image_id"]
        annotations_per_image.setdefault(img_id, []).append(ann)

    for img_id, anns in tqdm(annotations_per_image.items()):
        img_info = images[img_id]
        img_w, img_h = img_info["width"], img_info["height"]

        label_path = os.path.join(labels_dir, img_info["file_name"].replace(".jpg", ".txt"))

        with open(label_path, "w") as f:
            for ann in anns:
                x, y, w, h = ann["bbox"]

                x_center = (x + w / 2) / img_w
                y_center = (y + h / 2) / img_h
                w /= img_w
                h /= img_h

                class_id = categories[ann["category_id"]]

                f.write(f"{class_id} {x_center} {y_center} {w} {h}\n")

In [ ]:
#Apply conversion for training set
convert_coco_to_yolo(
    json_path="/content/compressed/annotations/instances_train.json",
    images_dir="/content/compressed/images/train",
    labels_dir="/content/compressed/labels/train"
)

In [ ]:
#Apply conversion for validation set
convert_coco_to_yolo(
    json_path="/content/compressed/annotations/instances_val.json",
    images_dir="/content/compressed/images/val",
    labels_dir="/content/compressed/labels/val"
)

In [ ]:
#Inspect train labels format
!ls /content/compressed/labels/train | head

In [ ]:
#Inspect train images format
!ls /content/compressed/images/train | head

In [ ]:
#Writing data.yaml for YOLO
%%writefile data.yaml
path: /content/compressed

train: images/train
val: images/val

names:
  0: swimmer
  1: boat
  2: jetski
  3: life_saving_appliances
  4: buoy

In [ ]:
#Install ultralytics package
!pip install ultralytics

In [ ]:
#Import libraries
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from google.colab import drive

In [ ]:
#Training baseline model
model = YOLO("yolov8s.pt")

model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="baseline_yolov8"
)

In [ ]:
#Mount Google Drive
drive.mount('/content/drive')

In [ ]:
#Copying folder into Drive for saving baseline model and related content
!cp -r /content/runs/detect/baseline_yolov8 /content/drive/MyDrive/

In [ ]:
#Load model back from Google Drive
WEIGHTS_PATH = "/content/drive/MyDrive/baseline_yolov8/weights/best.pt"
model = YOLO(WEIGHTS_PATH)

In [ ]:
#Evaluation (Validation)
metrics = model.val()
print(metrics)

In [ ]:
#Evaluation Metrics
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

In [ ]:
#Results.csv containing training and val info
df = pd.read_csv("/content/drive/MyDrive/baseline_yolov8/results.csv")
df.tail()

In [ ]:
#Plot validation mAP over Epochs
plt.plot(df["metrics/mAP50(B)"], label="mAP@0.5")
plt.plot(df["metrics/mAP50-95(B)"], label="mAP@0.5:0.95")
plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("Validation mAP over Epochs")
plt.legend()
plt.show()

In [ ]:
#Training vs Validation Box Loss over Epochs
df = pd.read_csv("/content/drive/MyDrive/baseline_yolov8/results.csv")

plt.plot(df["train/box_loss"], label="Train Box Loss")
plt.plot(df["val/box_loss"], label="Val Box Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Box Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#Test inference with 3 images
model.predict("/content/compressed/images/val/1003.jpg", save=True)
model.predict("/content/compressed/images/val/1018.jpg", save=True)
model.predict("/content/compressed/images/val/1032.jpg", save=True)



In [ ]:
#Display images
display(Image(filename="/content/compressed/images/val/1003.jpg"))
display(Image(filename="/content/compressed/images/val/1018.jpg"))
display(Image(filename="/content/compressed/images/val/1032.jpg"))

In [ ]:
#Display predicted images with class and bounding boxes
display(Image(filename="/content/runs/detect/predict/1003.jpg"))
display(Image(filename="/content/runs/detect/predict/1018.jpg"))
display(Image(filename="/content/runs/detect/predict/1032.jpg"))

In [ ]:
#Training the 1024 image input resolution model
model_1024 = YOLO("yolov8s.pt")

model_1024.train(
    data="data.yaml",
    epochs=50,
    imgsz=1024,
    batch=8,
    name="1024_imgsize_yolov8"
)

In [ ]:
#Mount drive and copy model and results to Drive
drive.mount('/content/drive')

!cp -r /content/runs/detect/1024_imgsize_yolov8 /content/drive/MyDrive/

In [ ]:
#Loading the 1024 model back from Drive
WEIGHTS_PATH_1024 = "/content/drive/MyDrive/1024_imgsize_yolov8/weights/best.pt"
model_1024 = YOLO(WEIGHTS_PATH_1024)

In [ ]:
#Validation and Evaluation metrics
metrics_1024 = model_1024.val()
print(metrics_1024)

print("mAP50:", metrics_1024.box.map50)
print("mAP50-95:", metrics_1024.box.map)
print("Precision:", metrics_1024.box.mp)
print("Recall:", metrics_1024.box.mr)

In [ ]:
#Results.csv
df_1024 = pd.read_csv("/content/drive/MyDrive/1024_imgsize_yolov8/results.csv")
df_1024.tail()

In [ ]:
#Validation mAP over Epochs
plt.plot(df_1024["metrics/mAP50(B)"], label="mAP@0.5")
plt.plot(df_1024["metrics/mAP50-95(B)"], label="mAP@0.5:0.95")

plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("Validation mAP over Epochs (1024)")
plt.legend()
plt.show()

In [ ]:
#Training vs Validation Box Loss over Epochs
plt.plot(df_1024["train/box_loss"], label="Train Box Loss")
plt.plot(df_1024["val/box_loss"], label="Val Box Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Box Loss (1024)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#Inference on images
model_1024.predict("/content/compressed/images/val/1003.jpg", save=True)
model_1024.predict("/content/compressed/images/val/1018.jpg", save=True)
model_1024.predict("/content/compressed/images/val/1032.jpg", save=True)

In [ ]:
#Display images
display(Image(filename="/content/compressed/images/val/1003.jpg"))
display(Image(filename="/content/compressed/images/val/1018.jpg"))
display(Image(filename="/content/compressed/images/val/1032.jpg"))

In [ ]:
#Display predicted image
display(Image(filename="/content/runs/detect/predict/1003.jpg"))
display(Image(filename="/content/runs/detect/predict/1018.jpg"))
display(Image(filename="/content/runs/detect/predict/1032.jpg"))

In [ ]:
#Install SAHI package
!pip install -U sahi

In [ ]:
#Loading baseline model into SAHI
from sahi import AutoDetectionModel

detection_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path="/content/drive/MyDrive/baseline_yolov8/weights/best.pt",
    confidence_threshold=0.25,
    device="cuda"
)

In [ ]:
#Running sliced detection on an image
from sahi.predict import get_sliced_prediction

result = get_sliced_prediction(
    "/content/compressed/images/val/1003.jpg",
    detection_model,
    slice_height=512,
    slice_width=512,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
)

result.export_visuals(export_dir="sahi_results/")

In [ ]:
#Running SAHI for validation images
from sahi.predict import predict

predict(
    model_type="yolov8",
    model_path="/content/drive/MyDrive/baseline_yolov8/weights/best.pt",
    source="/content/compressed/images/val",
    slice_height=512,
    slice_width=512,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    export_pickle=True,
    project="runs/sahi",
    name="val_results"
)

In [ ]:
#Preparing ground truth annotation for calculating
#COCO evaluation on SAHI model
import json
from pycocotools.coco import COCO

with open("/content/compressed/annotations/instances_val.json") as f:
    coco_gt_dict = json.load(f)

for ann in coco_gt_dict["annotations"]:
    if "iscrowd" not in ann:
        ann["iscrowd"] = 0

    if "area" not in ann:
        x, y, w, h = ann["bbox"]
        ann["area"] = w * h

coco_gt = COCO()
coco_gt.dataset = coco_gt_dict
coco_gt.createIndex()

print("Ground truth loaded and fixed")

In [ ]:
#Converting SAHI output into COCO eval for mAP and other metrics
import os
import pickle

pickle_dir = "runs/sahi/val_results3/pickles"

coco_results = []

filename_to_id = {
    img["file_name"]: img["id"]
    for img in coco_gt.dataset["images"]
}

for file_name in os.listdir(pickle_dir):
    if file_name.endswith(".pickle"):

        image_filename = file_name.replace(".pickle", ".jpg")
        image_id = filename_to_id[image_filename]

        with open(os.path.join(pickle_dir, file_name), "rb") as f:
            preds = pickle.load(f)

        for obj in preds:
            bbox = [float(x) for x in obj.bbox.to_xywh()]
            score = float(obj.score.value)

            category_id = int(obj.category.id) + 1

            coco_results.append({
                "image_id": int(image_id),
                "category_id": category_id,
                "bbox": bbox,
                "score": score
            })

print("Converted predictions:", len(coco_results))
print(coco_results[:3])

In [ ]:
#Print results
print(coco_results[:3])

In [ ]:
#Debug step to verify labels are correct
print("=== COCO GT categories ===")
for cat in coco_gt.dataset["categories"]:
    print(cat)

print("\n=== Example SAHI prediction category ===")
print("Pred category_id:", coco_results[0]["category_id"])

In [ ]:
#SAHI model evaluation metrics
import json
from pycocotools.cocoeval import COCOeval

with open("sahi_coco_results.json", "w") as f:
    json.dump(coco_results, f)

coco_dt = coco_gt.loadRes("sahi_coco_results.json")

coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

print("\n==== Summary Metrics ====")
print("mAP@0.5:0.95:", coco_eval.stats[0])
print("mAP@0.5:", coco_eval.stats[1])
print("Recall:", coco_eval.stats[8])

In [ ]:
#Function for several SAHI experiments
def run_sahi_experiment(name, slice_size, overlap):
    from sahi.predict import predict

    print(f"\nRunning {name}...")

    predict(
        model_type="yolov8",
        model_path="/content/drive/MyDrive/baseline_yolov8/weights/best.pt",
        source="/content/compressed/images/val",
        slice_height=slice_size,
        slice_width=slice_size,
        overlap_height_ratio=overlap,
        overlap_width_ratio=overlap,
        export_pickle=True,
        project="runs/sahi",
        name=name
    )

In [ ]:
#Run several SAHI experiments with different parameters
run_sahi_experiment("exp_512_02", 512, 0.2)
run_sahi_experiment("exp_256_02", 256, 0.2)
run_sahi_experiment("exp_512_03", 512, 0.3)

In [ ]:
#Evaluate all experiments
def evaluate_sahi(pickle_dir, save_name):
    import os, pickle, json
    from pycocotools.cocoeval import COCOeval

    coco_results = []

    filename_to_id = {
        img["file_name"]: img["id"]
        for img in coco_gt.dataset["images"]
    }

    for file_name in os.listdir(pickle_dir):
        if file_name.endswith(".pickle"):

            image_filename = file_name.replace(".pickle", ".jpg")
            image_id = filename_to_id[image_filename]

            with open(os.path.join(pickle_dir, file_name), "rb") as f:
                preds = pickle.load(f)

            for obj in preds:
                bbox = [float(x) for x in obj.bbox.to_xywh()]
                score = float(obj.score.value)

                category_id = int(obj.category.id) + 1

                coco_results.append({
                    "image_id": int(image_id),
                    "category_id": category_id,
                    "bbox": bbox,
                    "score": score
                })

    with open(save_name, "w") as f:
        json.dump(coco_results, f)

    coco_dt = coco_gt.loadRes(save_name)

    coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    return {
        "mAP50-95": coco_eval.stats[0],
        "mAP50": coco_eval.stats[1],
        "Recall": coco_eval.stats[8]
    }

In [ ]:
#Display evaluatoin results
results = {}

results["512_0.2"] = evaluate_sahi(
    "runs/sahi/exp_512_02/pickles",
    "runs/sahi/exp_512_02/sahi_results.json"
)

results["256_0.2"] = evaluate_sahi(
    "runs/sahi/exp_256_02/pickles",
    "runs/sahi/exp_256_02/sahi_results.json"
)

results["512_0.3"] = evaluate_sahi(
    "runs/sahi/exp_512_03/pickles",
    "runs/sahi/exp_512_03/sahi_results.json"
)

print("\nFinal Results:")
for k, v in results.items():
    print(k, v)

In [ ]:
#Save SAHI results as zip
!zip -r sahi_results.zip runs/sahi

In [ ]:
#Copy to Drive
!cp sahi_results.zip /content/drive/MyDrive/

In [ ]:
#Unzip in Google Drive
!unzip /content/drive/MyDrive/sahi_results.zip -d /content/drive/MyDrive/sahi_results

In [ ]:
#Define new architecture (Original YOLOv8 + CBAM in backbone)
#Convolutional Block Attention Module

%%writefile yolov8s_cbam.yaml

nc: 5

scales:
  n: [0.33, 0.25, 1024]
  s: [0.33, 0.50, 1024]
  m: [0.67, 0.75, 768]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.25, 512]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]

  - [-1, 1, CBAM, [128]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]

  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 7], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]

  - [[16, 19, 22], 1, Detect, [nc]]

In [ ]:
#Implementation of CBAM

import torch
import torch.nn as nn

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(hidden, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(
            self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x))
        )

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg, mx], dim=1)
        return self.sigmoid(self.conv(x))

class CBAM(nn.Module):
    def __init__(self, c1):
        super().__init__()
        self.ca = ChannelAttention(c1)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x

In [ ]:
#Add CBAM into YOLO's module registry
import ultralytics.nn.tasks as tasks

tasks.CBAM = CBAM

In [ ]:
#Load model based on custom YAML
model_cbam = YOLO("yolov8s_cbam.yaml")

In [ ]:
#Train CBAM model
model_cbam.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="cbam_yolov8"
)

In [ ]:
#Save model and results in Drive
!cp -r /content/runs/detect/cbam_yolov85 /content/drive/MyDrive/cbam_yolov8

In [ ]:
#Load model back from Drive

WEIGHTS_PATH_CBAM = "/content/drive/MyDrive/cbam_yolov8/weights/best.pt"

model_cbam = YOLO(WEIGHTS_PATH_CBAM)

In [ ]:
#Validation and evaluation metrics
metrics_cbam = model_cbam.val()

print("mAP50:", metrics_cbam.box.map50)
print("mAP50-95:", metrics_cbam.box.map)
print("Precision:", metrics_cbam.box.mp)
print("Recall:", metrics_cbam.box.mr)

In [ ]:
#Results.csv

df_cbam = pd.read_csv("/content/drive/MyDrive/cbam_yolov8/results.csv")
df_cbam.tail()

In [ ]:
#Plot CBAM Validation mAP over Epochs

plt.plot(df_cbam["metrics/mAP50(B)"], label="mAP@0.5")
plt.plot(df_cbam["metrics/mAP50-95(B)"], label="mAP@0.5:0.95")

plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("CBAM Validation mAP over Epochs")
plt.legend()
plt.show()

In [ ]:
#Training vs Validation Box Loss (CBAM)
plt.plot(df_cbam["train/box_loss"], label="Train Box Loss")
plt.plot(df_cbam["val/box_loss"], label="Val Box Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Box Loss (CBAM)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#Perform inference for CBAM model on sample images
model_cbam.predict("/content/compressed/images/val/1003.jpg", save=True)
model_cbam.predict("/content/compressed/images/val/1018.jpg", save=True)
model_cbam.predict("/content/compressed/images/val/1032.jpg", save=True)

In [ ]:
#Display images
display(Image(filename="/content/compressed/images/val/1003.jpg"))
display(Image(filename="/content/compressed/images/val/1018.jpg"))
display(Image(filename="/content/compressed/images/val/1032.jpg"))

In [ ]:
#Display predicted images

display(Image(filename="/content/runs/detect/predict/1003.jpg"))
display(Image(filename="/content/runs/detect/predict/1018.jpg"))
display(Image(filename="/content/runs/detect/predict/1032.jpg"))

In [ ]:
#Visual comparison between all three models

#Load model paths to load model from Drive

baseline_path = "/content/drive/MyDrive/baseline_yolov8/weights/best.pt"
model1024_path = "/content/drive/MyDrive/1024_imgsize_yolov8/weights/best.pt"
cbam_path = "/content/drive/MyDrive/cbam_yolov8/weights/best.pt"

baseline_model = YOLO(baseline_path)
model_1024 = YOLO(model1024_path)
model_cbam = YOLO(cbam_path)

In [ ]:
#Definte SAHI

from sahi import AutoDetectionModel

sahi_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path=baseline_path,
    confidence_threshold=0.25,
    device="cuda"
)

In [ ]:
#Selecting 12 random images to perform inference and visualize

import os
import random

val_dir = "/content/compressed/images/val"

images = sorted(os.listdir(val_dir))
sample_images = random.sample(images, 12)

sample_images[:5]

In [ ]:
#Perform inference on images with all 4 models

from sahi.predict import get_sliced_prediction

output_dir = "comparison_results"
os.makedirs(output_dir, exist_ok=True)

for img_name in sample_images:
    img_path = os.path.join(val_dir, img_name)

    print(f"Processing {img_name}...")

    baseline_model.predict(img_path, save=True, project=output_dir, name="baseline", exist_ok=True)

    model_1024.predict(img_path, save=True, project=output_dir, name="1024", exist_ok=True)

    model_cbam.predict(img_path, save=True, project=output_dir, name="cbam", exist_ok=True)

    result = get_sliced_prediction(
        img_path,
        sahi_model,
        slice_height=512,
        slice_width=512,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
    )

    result.export_visuals(
        export_dir=os.path.join(output_dir, "sahi"),
        file_name=img_name
    )

In [ ]:
#Organize results folder

!mv runs/detect/comparison_results/baseline /content/comparison_results/
!mv runs/detect/comparison_results/1024 /content/comparison_results/
!mv runs/detect/comparison_results/cbam /content/comparison_results/

In [ ]:
#Display a subset
#comparison between original image and model predictions


from IPython.display import Image, display

for img_name in sample_images[:1]:
    print(f"\n==== {img_name} ====")

    display(Image(filename=os.path.join(val_dir, img_name)))
    display(Image(filename=f"{output_dir}/baseline/{img_name}"))
    display(Image(filename=f"{output_dir}/1024/{img_name}"))
    display(Image(filename=f"{output_dir}/cbam/{img_name}"))
    display(Image(filename=f"{output_dir}/sahi/{img_name}.png"))

In [ ]:
import pkg_resources

packages = [
    "torch",
    "torchvision",
    "ultralytics",
    "pandas",
    "matplotlib",
    "opencv-python",
    "sahi",
    "pycocotools",
]

print("=== Installed Package Versions ===")
for pkg in packages:
    try:
        version = pkg_resources.get_distribution(pkg).version
        print(f"{pkg}=={version}")
    except:
        print(f"{pkg} not installed")